# 01 — EDA: clinical (synthetic cohort)

Distributional checks on synthetic clinical tables from `generate_synthetic_cohort`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koneke55/Mmvlm4SCD/blob/main/notebooks/01-eda-clinical.ipynb)

**Google Colab + GPU:** [Unsloth](https://unsloth.ai) documents a practical [Google Colab workflow](https://docs.unsloth.ai/get-started/install/google-colab) (free **T4** GPU tier, Runtime menu, run cells in order). Use it as the reference for attaching hardware acceleration.

**Note:** This repo does **not** depend on the `unsloth` pip package—only standard PyTorch + `pip install -e .`; the Unsloth guide covers Colab compute ergonomics.

**Local:** run `pip install -e .` from the repo root. **Colab:** run the environment cell below (clone under `/content` when needed).


## 1. Environment setup (Colab or local)

- **Colab:** optional `MMVLM_REPO_URL` for your fork; defaults to upstream.
- Installs this package editable (`pip install -e .`).


In [ ]:
import os
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "Could not find mmvlm4scd package root (missing src/mmvlm4scd). "
        "Open the notebook from the repo or run the Colab clone cell."
    )


if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/koneke55/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)


## 2. Accelerator check

Mirrors the GPU verification pattern recommended alongside [Unsloth's Colab instructions](https://docs.unsloth.ai/get-started/install/google-colab).


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime — for GPU follow Unsloth's Colab guide (Runtime → Change runtime type).")


## 3. Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mmvlm4scd.data import StandardPreprocessor, generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


## Load cohort and preprocess


In [ ]:
cfg = SCDSyntheticConfig(n_patients=1200, seed=42)
cohort = generate_synthetic_cohort(cfg)
clin: pd.DataFrame = cohort["clinical"]
clin.head()


In [ ]:
pre = StandardPreprocessor().fit(clin)
x_clin = pre.transform(clin)
print("Preprocessor output_dim (matches MultimodalSCDModel clinical_input_dim):", pre.output_dim)
print("Feature matrix shape:", x_clin.shape)


## Summary statistics


In [ ]:
clin.describe(include="all").T


## Genotype mix (Hb phenotypes)


In [ ]:
vc = clin["genotype"].value_counts(normalize=True).sort_index()
display(vc)
vc.plot(kind="bar", title="Genotype prevalence (synthetic)", rot=45)
plt.ylabel("fraction")
plt.tight_layout()
plt.show()


## Labs vs severity label


In [ ]:
sev = cohort["severity"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, ["hb_g_dl", "ldh_u_l", "voc_rate_per_year"]):
    for k in range(3):
        mask = sev == k
        ax.hist(clin.loc[mask, col].values, bins=20, alpha=0.45, label=f"sev {k}")
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
